# Week 1 (starter): Tokenization Analysis

This is the starter notebook for the Week 1 assignment. It runs as-is on placeholder text so you can see the shape of each step; your job is to replace the placeholders with your own passages and analysis, then commit it to your repository and open a pull request.

Cells marked **TODO (you)** are where you do the work. Everything runs in Jupyter or Google Colab. No GPU, no API key, one dependency: `tiktoken`.

The five parts match the assignment: stand up your repo, run the analysis, evaluate with real figures, find one failure, and submit.

In [21]:
# Setup. In Colab, uncomment the install line on first run.
# !pip install tiktoken
import os, pathlib
os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.') / '.tiktoken_cache').resolve())
os.makedirs(os.environ['TIKTOKEN_CACHE_DIR'], exist_ok=True)

import tiktoken
gpt4  = tiktoken.get_encoding('cl100k_base')   # GPT-4 / GPT-3.5
gpt4o = tiktoken.get_encoding('o200k_base')    # GPT-4o
print('tiktoken', tiktoken.__version__, '- encoders ready (cl100k_base, o200k_base)')

tiktoken 0.14.0 - encoders ready (cl100k_base, o200k_base)


## Part 1: Stand up your repository

Do this once, outside the notebook:

1. Create a public repo (suggested name `cosc-650`).
2. Add a `README.md` a stranger could read (what it is, how it is organized, the tools you use).
3. Add an agent context file that your AI tool reads, with project context and conventions. `AGENTS.md` is the cross-tool convention; `CLAUDE.md` and `GEMINI.md` are tool-specific variants. Use whichever your tool reads.
4. Work on a branch and open a pull request into `main`. You will do this every week.

Then commit this notebook into the repo and keep going.

## Helpers (provided)

Two small functions: count tokens for a string, and show the exact sub-token pieces a word breaks into. The demo uses a line you may recognize.

In [22]:
def count_tokens(text, enc):
    return len(enc.encode(text))

def show_split(word, enc=gpt4):
    ids = enc.encode(word)
    pieces = [enc.decode([i]) for i in ids]
    print(f'{word!r:18s} -> {len(ids)} token(s): {pieces}')

# demo: some short strings are a single token; capitalized or rarer words fragment
for w in ['Panic', ' towel', '42', 'antidisestablishmentarianism']:
    show_split(w)

'Panic'            -> 2 token(s): ['P', 'anic']
' towel'           -> 1 token(s): [' towel']
'42'               -> 1 token(s): ['42']
'antidisestablishmentarianism' -> 6 token(s): ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']


## Part 2: Your passages

**TODO (you):** replace the two placeholders with your own text. The non-English passage must be at least 100 words, with a faithful English translation. The placeholders below are short Hitchhiker's Guide lines so the notebook runs; swap in your real passages.

In [23]:
english_text = "Artificial intelligence is becoming an increasingly important part of modern software systems. Developers use artificial intelligence to analyze information, automate repetitive tasks, generate text, and help users make decisions. Large language models are especially interesting because they can understand and generate natural language across many different topics. However, these models do not process sentences in the same way that humans do. Before text can be processed by a language model, it must be divided into smaller units called tokens. A token may represent an entire word, part of a word, punctuation, or another sequence of characters. The way a tokenizer divides text can vary significantly between languages. As a result, two passages that communicate the same information may require very different numbers of tokens."
foreign_text = "La inteligencia artificial se está convirtiendo en una parte cada vez más importante de los sistemas de software modernos. Los desarrolladores utilizan la inteligencia artificial para analizar información, automatizar tareas repetitivas, generar texto y ayudar a los usuarios a tomar decisiones. Los modelos de lenguaje grandes son especialmente interesantes porque pueden comprender y generar lenguaje natural sobre muchos temas diferentes. Sin embargo, estos modelos no procesan las oraciones de la misma manera que los seres humanos. Antes de que un modelo de lenguaje pueda procesar un texto, este debe dividirse en unidades más pequeñas llamadas tokens. Un token puede representar una palabra completa, una parte de una palabra, un signo de puntuación u otra secuencia de caracteres. La forma en que un tokenizador divide el texto puede variar significativamente entre idiomas. Como resultado, dos textos que comunican la misma información pueden requerir cantidades muy diferentes de tokens."

print('English words:', len(english_text.split()))
print('Foreign words:', len(foreign_text.split()))

def report(label, text):
    print(f'{label:9s} | chars {len(text):4d} | GPT-4 {count_tokens(text, gpt4):4d} | GPT-4o {count_tokens(text, gpt4o):4d}')

report('English', english_text)
report('Foreign', foreign_text)

tax_gpt4  = count_tokens(foreign_text, gpt4)  / count_tokens(english_text, gpt4)
tax_gpt4o = count_tokens(foreign_text, gpt4o) / count_tokens(english_text, gpt4o)
print(f'\nMultilingual tax  GPT-4: {tax_gpt4:.2f}x   GPT-4o: {tax_gpt4o:.2f}x')
# I chose Spanish as my second language. The Spanish text had more words and required more
# tokens than the equivalent English text across both models. The multilingual tax was 1.46x
# for GPT-4 and 1.25x for GPT-4o, meaning the Spanish text required about 46% and 25% more tokens,
# respectively. This also shows that GPT-4o tokenizes Spanish more efficiently than GPT-4.


English words: 124
Foreign words: 147
English   | chars  836 | GPT-4  142 | GPT-4o  141
Foreign   | chars  997 | GPT-4  207 | GPT-4o  176

Multilingual tax  GPT-4: 1.46x   GPT-4o: 1.25x


## Part 3: Evaluate with real figures

Turn the counts into engineering consequences. The skeleton below computes both; keep it pointed at your real passages.

In [24]:
CTX = 128_000
en = count_tokens(english_text, gpt4)
fo = count_tokens(foreign_text, gpt4)
print(f'A {CTX:,}-token window holds about {CTX//en:,} English copies and {CTX//fo:,} foreign copies of your passage.')
print(f'Per-request cost multiplier for the foreign language: {fo/en:.2f}x (billing is per token).')

# The 128,000-token context window could fit the English text about 901 times,
# whereas the Spanish text could fit about 618 times. For a product serving
# Spanish-speaking users, this means application owners may have higher costs
# to process the same content in Spanish compared to English because the Spanish text requires more tokens.

A 128,000-token window holds about 901 English copies and 618 foreign copies of your passage.
Per-request cost multiplier for the foreign language: 1.46x (billing is per token).


## Part 4: Bias splits and one failure

**TODO (you):** (a) pick three words where your non-English form fragments far worse than the English equivalent, and show both with `show_split`; (b) find ONE input whose token count defies intuition and explain it. A few failure candidates are demonstrated below to get you started; replace them with your own find and write the explanation plus a mitigation.

In [25]:
# (a) Three English/Spanish bias pairs

print('Pair 1: Knowledge / Conocimiento')
show_split('knowledge')
show_split('conocimiento')

print('\nPair 2: Research / Investigación')
show_split('research')
show_split('investigación')

print('\nPair 3: Trust / Confianza')
show_split('trust')
show_split('confianza')


# (b) ONE input whose token count defies intuition and explain it
show_split('andrew.clark@example.com')

# Explanation
### (a) Bias Splits

# I chose three words to compare the tokenization of English versus Spanish: knowledge, research, and trust.
# All three of their Spanish equivalents were broken into 3 tokens, whereas their English counterparts only
# required 1 token. For example, the English word "knowledge" was compressed into 1 token, whereas its
# Spanish counterpart, "conocimiento," was split into 3 tokens: `'con'`, `'oc'`, and `'imiento'`.
# This increase in tokens for equivalent Spanish words demonstrates a multilingual bias or tax that can
# make processing some languages more expensive and consume more of the model's context window.

### (b) Email Failure

# I processed an email address (`andrew.clark@example.com`) and expected it to produce only 1–2 tokens.
# Instead, it generated 6 tokens: `'and'`, `'rew'`, `'.cl'`, `'ark'`, `'@example'`, and `'.com'`.
# This result was surprising because an email address appears to be one compact piece of information,
# but the tokenizer breaks it into smaller character patterns. This can cause email addresses and other
# structured identifiers to consume more tokens than expected. One potential mitigation for this issue
# is cleaning or replacing unnecessary email addresses with shorter placeholders before processing.


Pair 1: Knowledge / Conocimiento
'knowledge'        -> 1 token(s): ['knowledge']
'conocimiento'     -> 3 token(s): ['con', 'oc', 'imiento']

Pair 2: Research / Investigación
'research'         -> 1 token(s): ['research']
'investigación'    -> 3 token(s): ['invest', 'ig', 'ación']

Pair 3: Trust / Confianza
'trust'            -> 1 token(s): ['trust']
'confianza'        -> 3 token(s): ['conf', 'ian', 'za']
'andrew.clark@example.com' -> 6 token(s): ['and', 'rew', '.cl', 'ark', '@example', '.com']


## Part 5: Submit

Before you open the pull request, check:

- The notebook runs top to bottom on **your** passages, not the placeholders.
- Your three bias splits are shown and explained.
- The failure case has a cause and a mitigation.
- The PR description has a one-paragraph result summary with your headline numbers.
- You linked one issue in your repo logging this as a research note (title, inputs, what you found).

Rubric: repo quality (15), counts from both tokenizers (20), tax computed (15), three bias splits (20), cost and context figures (15), the failure case (10), PR hygiene (5).